### Imports and Setup
Explores Claude API integration for automatic transaction categorization.
Tests prompt design and identifies miscategorized transactions.

In [1]:
import anthropic
import os
import json
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../backend/.env")

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

In [6]:
CATEGORIES = [
    "Income", "Groceries", "Dining", "Coffee",
    "Transportation", "Shopping", "Subscriptions",
    "Health & Wellness", "Transfer","Entertainment", "Other"
]

# new prompt to test
SYSTEM_PROMPT = f"""You are a financial transaction categorizer for a Canadian personal finance app.

You will receive a bank transaction description and an optional transaction code.
Transaction codes mean:
- DN: direct deposit (likely Income or Transfer)
- CW: cash withdrawal or e-transfer sent (likely Transfer)
- PR: point of sale purchase (likely Shopping, Groceries, Dining, Coffee etc.)
- OP: online purchase (likely Shopping, Subscriptions etc.)
- DS: direct payment (likely Subscriptions or Bills)
- RN: retail purchase (likely Shopping or Dining)

Respond ONLY with a JSON object in this exact format: {{"category": "CategoryName"}}
Choose from this exact list: {CATEGORIES}

Rules:
- Return only valid JSON, no explanation, no markdown, no extra text
- If uncertain, return "Other" — never return null
- Focus on the merchant name at the START of the description, ignore store numbers and locations

Canadian specific rules:
- TLNK or TRANSLINK or COMPASS = Transportation
- UBER or UBERTRIP = Transportation
- CRA or CANADA REVENUE = Income
- BC REVENUE or REVENUE SERVICES BC = Health & Wellness
- TF followed by numbers = Transfer
- INTERAC ETRNSFR = Transfer
- BELL MOBILITY or TELUS or ROGERS or FIDO = Subscriptions
- DYNAMITE or GARAGE or ARITZIA or LULULEMON = Shopping
- CACTUS CLUB or GLOWBAL or MCDONALD or SUBWAY = Dining
- CHARTWELLS or SFU = Dining
- STARBUCKS or TIM HORTONS or BLENZ = Coffee
- WALMART or SUPERSTORE or SAVE ON or SAFEWAY or FRESHCO = Groceries
- SHOPPERS or LONDON DRUGS or PHARMASAVE = Health & Wellness
- FLOWER or FLORAL or FLORIST = Shopping

Example:
Input: [PR] TIM HORTONS VANCOUVER BC
Output: {{"category": "Coffee"}}"""

In [ ]:
import importlib
import sys

# remove cached version
if "backend.pipeline.categorization" in sys.modules:
    del sys.modules["backend.pipeline.categorization"]
if "pipeline.categorization" in sys.modules:
    del sys.modules["pipeline.categorization"]

# fresh import
from backend.pipeline.categorization import categorize_all_pending, categorize_transaction
# Clear all the categories to NULL
with engine.connect() as connection:
    connection.execute(text("UPDATE transactions SET category = NULL"))
    connection.commit()
    print("All categories reset")

# rerun categorization
categorize_all_pending(engine, client)

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("""
    SELECT description, transaction_code, category 
    FROM transactions 
    WHERE category = 'Other'
    ORDER BY description
    """))
    rows = result.fetchall()
print(rows)

In [ ]:
import pandas as pd
df = pd.read_sql("""
    SELECT description, transaction_code, category 
    FROM transactions 
    WHERE category = 'Other'
    ORDER BY description
""", engine)
print(df.to_string())